In [0]:
%sql
-- Check whether Databricks can list the article files written by Lambda.
LIST 's3://databricks-news-articles/kontrast.at/' LIMIT 3;

In [0]:
-- Read one article and preview the first 500 characters of its HTML.
SELECT
    path,
    length AS size_bytes,
    substring(decode(content, 'UTF-8'), 1, 500) AS html_preview
FROM read_files(
    's3://databricks-news-articles/kontrast.at/1-mai-wehren-wir-uns-gegen-den-klassenkampf-von-oben.html',
    format => 'binaryFile'
);

In [0]:
%python
# Parse one archived article and show its source path with extracted fields.
from parse_article import parse_article

article_path = 's3://databricks-news-articles/kontrast.at/1-mai-wehren-wir-uns-gegen-den-klassenkampf-von-oben.html'
row = spark.read.format('binaryFile').load(article_path).select('path', 'content').first()
article = parse_article(row.content.decode('utf-8', errors='replace'))

print('SOURCE FILE:', row.path)
for field, value in article.items():
    print(f'{field.upper()}: {value}')

In [0]:
%python
# Build one typed Silver candidate from the parsed article and S3 path.
from datetime import datetime
from pyspark.sql.types import ArrayType, StringType, StructField, StructType, TimestampType

author_schema = StructType([StructField('name', StringType()), StructField('profile_url', StringType())])
link_schema = StructType([StructField('text', StringType()), StructField('url', StringType())])
schema = StructType([
    StructField('canonical_url', StringType()),
    StructField('headline', StringType()),
    StructField('body_text', StringType()),
    StructField('published_at', TimestampType()),
    StructField('modified_at', TimestampType()),
    StructField('authors', ArrayType(author_schema)),
    StructField('tags', ArrayType(StringType())),
    StructField('categories', ArrayType(StringType())),
    StructField('links', ArrayType(link_schema)),
    StructField('source_path', StringType()),
])

# Convert available publisher dates to timestamps and include the original file path.
record = dict(article)
record['published_at'] = datetime.fromisoformat(article['published_at'])
record['modified_at'] = datetime.fromisoformat(article['modified_at']) if article['modified_at'] else None
record['source_path'] = row.path
silver_candidate = spark.createDataFrame([record], schema=schema)
silver_candidate.printSchema()
display(silver_candidate)

In [0]:
%python
# Parse three archived HTML files into typed Silver candidate rows.
from datetime import datetime
from pyspark.sql.functions import udf

sample_paths = [
    's3://databricks-news-articles/kontrast.at/3-gruende-warum-sich-parteien-nicht-von-privaten-investoren-finanzieren-lassen-sollten.html',
    's3://databricks-news-articles/kontrast.at/achim-truger-kurt-rothschild-preis.html',
    's3://databricks-news-articles/kontrast.at/arme-familien-in-oesterreich-2022.html',
]

# Decode each file and return the parsed fields with its S3 path.
def parse_file(path, content):
    # Include the file path when article parsing fails.
    try:
        result = parse_article(content.decode('utf-8', errors='replace'))
    except ValueError as exc:
        raise ValueError(f'{path}: {exc}') from exc
    # Leave non-article pages out of the parsed rows.
    if result is None:
        return None
    result['published_at'] = datetime.fromisoformat(result['published_at'])
    result['modified_at'] = datetime.fromisoformat(result['modified_at']) if result['modified_at'] else None
    result['source_path'] = path
    return result

parse_file_udf = udf(parse_file, schema)
source_files = spark.read.format('binaryFile').load(sample_paths)
silver_batch = source_files.select(parse_file_udf('path', 'content').alias('article')).select('article.*')
silver_batch.printSchema()
display(silver_batch)

In [0]:
%python
# Parse the HTML archive and compare file, article, and URL counts.
from pyspark.sql import functions as F

archive_path = 's3://databricks-news-articles/kontrast.at/'
source_files = spark.read.format('binaryFile').option('pathGlobFilter', '*.html').load(archive_path)
parsed_files = source_files.select('path', parse_file_udf('path', 'content').alias('article'))
silver_all = parsed_files.filter(F.col('article').isNotNull()).select('article.*')
skipped_files = parsed_files.filter(F.col('article').isNull()).select('path')

print('HTML FILES:', source_files.count())
display(silver_all.agg(
    F.count('canonical_url').alias('parsed_articles'),
    F.countDistinct('canonical_url').alias('distinct_urls'),
))
print('SKIPPED PAGES:', skipped_files.count())
display(skipped_files)